In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors flask pyngrok
!pip install -q huggingface_hub

In [ ]:
# Cell 2: Download model + start API server
import torch
from diffusers import StableDiffusionPipeline
from flask import Flask, request, send_file, jsonify
from pyngrok import ngrok
import io, os, threading, time

# Load anime NSFW model (Anything V5 - well-known anime model)
print('Loading model...')
model_id = 'stablediffusionapi/anything-v5'
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()
print('Model loaded!')

# Flask API
app = Flask(__name__)

@app.route('/generate', methods=['POST'])
def generate():
    data = request.json
    prompt = data.get('prompt', '')
    negative = data.get('negative_prompt', 'ugly, deformed, blurry, low quality, bad anatomy, censored, clothed, text, watermark')
    steps = min(data.get('steps', 25), 50)
    width = data.get('width', 512)
    height = data.get('height', 512)
    
    # Add quality boosters to prompt
    full_prompt = f'{prompt}, masterpiece, best quality, detailed, anime, hentai'
    
    image = pipe(
        full_prompt,
        negative_prompt=negative,
        num_inference_steps=steps,
        guidance_scale=7.5,
        width=width,
        height=height
    ).images[0]
    
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    buf.seek(0)
    return send_file(buf, mimetype='image/png')

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'model': model_id})

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print(f'\n🔥 API is live at: {public_url}')
print(f'\nGive this URL to the web app!')

# Run Flask
app.run(port=5000)